# Interest Rates - ARIMA Forecasting Pipeline
## 1. Imports
Loading all the required libraries for our time series analysis and modeling.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pmdarima import auto_arima
%matplotlib inline


## 2. Load Dataset
Loading the raw dataset. Handling any potential skipped rows common in World Bank datasets.


In [ ]:
file_path = 'Data csvs/interest_rates.csv'
# Try loading normally first, if columns don't look right, try skipping rows
try:
    df_raw = pd.read_csv(file_path)
    if 'Country Name' not in df_raw.columns and 'Country Code' not in df_raw.columns and df_raw.shape[1] < 3:
        df_raw = pd.read_csv(file_path, skiprows=4)
except Exception:
    df_raw = pd.read_csv(file_path, skiprows=4)
df_raw.columns = df_raw.columns.str.strip()
print('Columns:', df_raw.columns)
print(df_raw.head())


## 3. Data Cleaning & Formatting
Filtering the data for India, removing unnecessary columns, handling missing values, reshaping the dataset, and setting the Year as a datetime index.


In [ ]:
# Find country column (case-insensitive)
country_col = next((col for col in df_raw.columns if col.lower() in ['country name', 'country']), None)
if not country_col:
    raise ValueError('Could not find Country column')

# Filter for India
df_india = df_raw[df_raw[country_col].astype(str).str.upper() == 'INDIA'].copy()

# Drop non-year columns
cols_to_drop = [col for col in df_raw.columns if not col.isdigit()]
df_india_values = df_india.drop(columns=cols_to_drop)

# Melt the dataframe to have Year and Value columns
df_india_Values_melted = df_india_values.melt(
    var_name='Year',
    value_name='Value'
)

# Convert Year to int, sort, and convert to datetime
df_india_Values_melted['Year'] = df_india_Values_melted['Year'].astype(int)
df_india_Values_melted = df_india_Values_melted.sort_values('Year').reset_index(drop=True)
df_india_Values_melted['Year'] = pd.to_datetime(df_india_Values_melted['Year'], format='%Y')
df_india_Values_melted = df_india_Values_melted.set_index('Year')

# Handle Missing Values: Interpolate and drop remaining NAs
df_india_Values_melted['Value'] = pd.to_numeric(df_india_Values_melted['Value'], errors='coerce')
df_india_Values_melted['Value'] = df_india_Values_melted['Value'].interpolate(method='linear').ffill().bfill()
df_india_Values_melted = df_india_Values_melted.dropna()

print('Cleaned Data Shape:', df_india_Values_melted.shape)
print(df_india_Values_melted.head())

# Save the cleaned dataset
df_india_Values_melted.to_csv('../data/processed/interest_rates_india.csv')


## 4. Exploratory Data Analysis (EDA)
Plotting the time series.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_india_Values_melted.index, df_india_Values_melted['Value'], color='blue')
plt.title('Interest Rates in India Over Time')
plt.xlabel('Year')
plt.ylabel('Value')
plt.grid(True)
plt.show()


## 5. Stationarity Testing (ADF Test)
Using the Augmented Dickey-Fuller test to determine if the series is stationary.


In [ ]:
result = adfuller(df_india_Values_melted['Value'])
print('ADF Statistic:', result[0])
print('p-value:', result[1])
if result[1] <= 0.05:
    print('The time series is likely stationary (Reject the null hypothesis)')
else:
    print('The time series is likely non-stationary (Fail to reject the null hypothesis)')


## 6. Differencing and ACF/PACF
Visualizing Autocorrelation and Partial Autocorrelation. If the series was non-stationary, we apply differencing before plotting.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# Original Data
plot_acf(df_india_Values_melted['Value'], ax=axes[0, 0], title='ACF (Original)')
plot_pacf(df_india_Values_melted['Value'], ax=axes[0, 1], title='PACF (Original)')

# First Differenced Data
diff_data = df_india_Values_melted['Value'].diff().dropna()
plot_acf(diff_data, ax=axes[1, 0], title='ACF (1st Difference)')
plot_pacf(diff_data, ax=axes[1, 1], title='PACF (1st Difference)')

plt.tight_layout()
plt.show()


## 7. Train / Test Split
Splitting the data chronologically:
- Train: 1991–2018
- Test: 2019–2024


In [ ]:
# Filter the dataframe to only include indices from 1991 onwards
df_filtered = df_india_Values_melted.loc['1991':]

train = df_filtered.loc[:'2018']
test = df_filtered.loc['2019':'2024']

print('Train shape:', train.shape)
print('Test shape:', test.shape)


## 8. Auto ARIMA Model Selection
Using auto_arima to automatically find the best (p,d,q) parameters based on the lowest AIC score.


In [ ]:
auto_model = auto_arima(
    train['Value'],
    start_p=0, start_q=0,
    max_p=5, max_q=5,
    d=None, # let auto_arima find 'd'
    seasonal=False,
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)
print(auto_model.summary())
best_order = auto_model.order
print(f'\nBest ARIMA Order Selected: {best_order}')


## 9. Train Final ARIMA Model
Training the final ARIMA model using the optimal order identified by Auto ARIMA.


In [ ]:
model = ARIMA(train['Value'], order=best_order)
results = model.fit()
print(results.summary())


## 10. Forecast (2019–2024)
Predicting the values for the test period.


In [ ]:
forecast = results.forecast(steps=len(test))
comparison = test.copy()
comparison['Predicted'] = forecast
print(comparison)


## 11. Model Evaluation (MAE, RMSE)
Evaluating the forecast accuracy against actual unseen data.


In [ ]:
mae = mean_absolute_error(comparison['Value'], comparison['Predicted'])
rmse = np.sqrt(mean_squared_error(comparison['Value'], comparison['Predicted']))
print('Mean Absolute Error (MAE):', mae)
print('Root Mean Squared Error (RMSE):', rmse)


## 12. Visualization (Actual vs Predicted)
Plotting the training data, actual test data, and our predictions.


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(train.index, train['Value'], label='Training Data', color='blue')
plt.plot(test.index, test['Value'], label='Actual Test Data', color='orange')
plt.plot(test.index, comparison['Predicted'], label='Predicted', linestyle='--', color='red')
plt.title('Interest Rates Forecast vs Actuals')
plt.xlabel('Year')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()


## 13. Residual Diagnostics
Checking if residuals behave like random noise.
- Plot residuals
- Plot ACF of residuals
- Run Ljung-Box test


In [ ]:
residuals = results.resid

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].plot(residuals)
axes[0].set_title('Residuals of ARIMA Model')
axes[0].grid(True)

plot_acf(residuals, ax=axes[1], title='ACF of Residuals')

plt.show()

lb_test = acorr_ljungbox(residuals, lags=[10], return_df=True)
print('\nLjung-Box Test Results:')
print(lb_test)


## 14. Save Results
Export the predictions to a CSV and save the model as a pickle (.pkl) file.


In [ ]:
import os

os.makedirs('../results/', exist_ok=True)
os.makedirs('../models/', exist_ok=True)

comparison.to_csv('../results/interest_rates_predictions.csv')
with open('../models/interest_rates_arima_model.pkl', 'wb') as f:
    pickle.dump(results, f)
print('Predictions and model saved successfully!')


In [ ]:
import pandas as pd
# Generate Backtest CSV
backtest = comparison.copy()
backtest['Value_diff'] = backtest['Value'].diff()
backtest.to_csv('../results/interest_rates_backtest.csv')
print('Backtest saved successfully!')


In [ ]:
# Generate Component Report
report_content = f"""# ESI Report Component: Interest Rates\n\n## Dataset\n- **Indicator**: Interest Rates\n- **Period**: {df_india_Values_melted.index.min().year}–{df_india_Values_melted.index.max().year}\n- **Source**: RBI / MOSPI / World Bank\n\n## Model\n- **Algorithm**: ARIMA(p,d,q)\n- **Selection Method**: Hyperparameters selected via Auto-ARIMA\n\n## Evaluation\n- **Metrics**: \n  - Mean Absolute Error (MAE)\n  - Root Mean Squared Error (RMSE)\n- **Backtest Period**: {test.index.min().year}–{test.index.max().year}\n\n## Visualizations\n- **Actual vs Forecast**: Demonstrates model fit and accuracy over the testing period.\n- **Residual Plot**: Verifies that residuals resemble random noise with no visible patterns.\n"""

with open('../results/Interest_Rates_Component_Report.md', 'w') as f:
    f.write(report_content)
print('Component report successfully saved to ../results/Interest_Rates_Component_Report.md')
